<a href="https://colab.research.google.com/github/Zain506/MedCLIP-SAM/blob/main/notebooks/ImageSegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Segmentation

**MedCLIP-SAM**
[Paper](https://arxiv.org/pdf/2403.20253)


**gScoreCAM**
[Paper](https://www.google.com/url?q=https%3A%2F%2Fopenaccess.thecvf.com%2Fcontent%2FACCV2022%2Fpapers%2FChen_gScoreCAM_What_objects_is_CLIP_looking_at_ACCV_2022_paper.pdf)

In [ ]:
%pip install open-clip-torch -q

In [2]:
from google.colab import drive

drive.mount("/content/drive/")

Mounted at /content/drive/


## Load BiomedCLIP

In [ ]:
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
weights_path = "/content/drive/MyDrive/colab/MedCLIP-SAM/biomedclip_weights.pth"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device)

In [4]:
# We are looking for the final attention layer
# We place a hook in it to retrieve
modules = dict(model.named_modules())
print(modules["visual"].transformer.resblocks[10].attn) # For full model, print(modules[""])

MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
)


# Define a method to view the activations and gradients

In [11]:
def online_test(img, text, layer=-2): # Run on 1 data point
    activations = {}

    def forward_hook(module, inp, out):
        out = out[0]
        out.retain_grad() # Save gradient
        activations["attn"] = out # Store

    # 2nd last attention layer: Default to 2nd last
    layer = model.visual.transformer.resblocks[layer].attn
    handle_fwd = layer.register_forward_hook(forward_hook) # Register hook

    # forward
    img_emb = model.encode_image(img)
    text_emb = model.encode_text(text)

    # differentiable loss
    sim = img_emb @ text_emb.T
    loss = -sim.diag().mean() # Temporary loss function

    loss.backward() # Backward pass

    handle_fwd.remove() # Remove hook

    act = activations["attn"] # Retrieve activations
    grad = act.grad # Retrieve gradients

    return act, grad, loss


# Load data to test

In [ ]:
from datasets import load_dataset
ds = load_dataset("adishourya/MEDPIX-ClinQA") # Same MedPIX dataset that fine-tuned MedCLIP
train_valid = ds["train"].train_test_split(test_size=0.1)
training = train_valid["train"].select(range(100))
test = train_valid["test"]

In [7]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def getData(i): # Retrieve data and generate initial embeddings
  text = tokenizer(training[i]["question"] + "\n" + training[0]["answer"])
  image = preprocess(training[i]["image_id"]).unsqueeze(0).to(device)

  return text, image


In [33]:
text, image = getData(0) # Run data retrieval and tokenisation
tmp = online_test(image, text) # Retrieve 2nd last attention layer
final_layer = online_test(image, text, layer=-1) # Retrieve final attention layer

In [45]:
def visualise(obj): # Return tensors representing attention layer obj
  print("Activations".center(50, " "))
  obj1 = obj[0]
  print(obj1.shape)
  print(obj1)
  print("\n"*2)
  print("Gradients".center(50, " "))
  obj2 = obj[1]
  print(obj2.shape)
  print(obj2)
  print("\n"*2)
  print("Loss".center(50, " "))
  print("\n")
  print(obj[2])

In [44]:
# visualise(final_layer)
# print(final_layer[1][0].shape)
# print(final_layer[1][0])
verifier = (final_layer[1][0] == 0).all(dim=1) # This vector checks if every element in the Jacobian (derivative) vector is zero
print(verifier.shape) # 50 vectors each True or False checking if it is filled with zeroes or not
print(verifier) # Only the first one is False, in line with the theory that the first token is the CLS
# The gradients in the final layer are all 0 except the first one
# Therefore we verify the first token is the CLS token and should be removed

torch.Size([50])
tensor([False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True])
